# FRA red-team model comparison

Executed in Google Colab on 2026-10-30. Frozen v4 source: 44d5abd3363248b1de0a3ed96ff32982aafdd7b3. Compare unchanged baseline with Extra Trees and a predeclared equal-weight blend. October is an exploratory holdout already used in development; this is not a v5 benchmark or a production promotion. Upload fra-red-team-source.zip when running setup.

50 existing tests passed. Archive SHA-256 was verified before extraction. Per-row traces retain features, labels, split timestamps, exclusions and all candidate probabilities.

In [2]:
import os, sys, subprocess, pathlib, hashlib, json
REV = "44d5abd3363248b1de0a3ed96ff32982aafdd7b3"
ROOT = pathlib.Path("/content/fra-red-team")
from google.colab import files
import zipfile, io
uploaded = files.upload()
archive = uploaded["fra-red-team-source.zip"]
assert hashlib.sha256(archive).hexdigest() == "74f91c3a19fb545989f48a5bd0478a5915529ff797d09addfead48b117ea6b18"
ROOT.mkdir(exist_ok=True)
with zipfile.ZipFile(io.BytesIO(archive)) as bundle:
    bundle.extractall(ROOT)
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "ml/requirements.txt"], check=True)
OUT = pathlib.Path("/content/fra-red-team-results")
OUT.mkdir(exist_ok=True)
result = subprocess.run([sys.executable, "-m", "pytest", "ml/tests", "-q"], text=True, capture_output=True)
(OUT / "pytest.txt").write_text(result.stdout + result.stderr)
print(result.stdout, result.stderr)
assert result.returncode == 0, "Existing tests failed"
print("SETUP_AND_EXISTING_TESTS_PASSED", REV)

..................................................                       [100%]
50 passed in 2.76s


In [3]:
# Controlled October red-team experiment: same files, features, split and scorer.
assert result.returncode == 0
import numpy as np, pandas as pd, sklearn, joblib, platform, time
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from ml import train as training
from ml.features import FEATURES
from ml.predict import probability, od_probability
from ml.quality import skill
from threadpoolctl import threadpool_limits
started = time.time()
threadpool_limits(limits=2)
frame, truth, obs, skipped = training.dataset(stride=6)
start, end = pd.Timestamp("2026-10-01", tz="UTC"), pd.Timestamp("2026-11-01", tz="UTC")
train_rows = frame.loc[frame.label_available_at < start].copy()
test_rows = frame.loc[(frame.issue_time >= start) & (frame.issue_time < end) & (frame.label_available_at < end)].copy()
fit_rows = training.without_stuck_feed(train_rows)
assert len(fit_rows) and len(test_rows)
assert fit_rows.label_available_at.max() < test_rows.issue_time.min()
assert not set(FEATURES) & {"y_od", "y_r18", "suspect_target", "suspect_input", "changed_within_lead"}
print("ROWS", len(fit_rows), len(test_rows), "FEATURES", len(FEATURES), flush=True)
print("Refitting original model and its chronological persistence blend", flush=True)
original = training.fit(fit_rows)
print("Fitting alternative Extra Trees on identical labelled rows", flush=True)
labelled = fit_rows.dropna(subset=["y_od"])
alternate = make_pipeline(SimpleImputer(strategy="median", add_indicator=True, keep_empty_features=True), ExtraTreesClassifier(n_estimators=100, max_depth=15, min_samples_leaf=50, max_features=1.0, n_jobs=2, random_state=42))
alternate.fit(labelled[FEATURES], labelled.y_od.astype(int))
raw = probability(original["od"], test_rows[FEATURES])
blend = od_probability(original, test_rows)
extra = probability(alternate, test_rows[FEATURES])
predictions = {"original_raw": raw, "original_blend": blend, "extra_trees": extra, "equal_mix": 0.5 * blend + 0.5 * extra, "persistence": test_rows.od_at_issue.to_numpy(float), "rules": test_rows.rule_od25.to_numpy(float)}
for values in predictions.values():
    assert len(values) == len(test_rows) and np.isfinite(values).all()
    assert ((values >= 0) & (values <= 1)).all()
# All methods share precisely the same valid-label population.
common = test_rows[["y_od", "rule_od25", "od_at_issue"]].notna().all(axis=1)
scored = test_rows.loc[common].copy()
records = []
for name, values in predictions.items():
    scored[name] = values[common.to_numpy()]
    candidate = scored.assign(p_od25=scored[name], p_od25_raw=scored[name])
    slices = {"all_recorded": candidate, "suspect_excluded_sensitivity": training.without_stuck_feed(candidate), "midnight_issues": candidate.loc[candidate.issue_time.dt.tz_convert("Europe/Berlin").dt.hour.eq(0)]}
    for slice_name, subset in slices.items():
        for row in skill(subset, "od"):
            records.append(dict(method=name, slice=slice_name, lead_h=row["lead_h"], n=row["n"], accuracy=row["model_acc"], brier=row["model_brier"], changed_n=row["change_n"], changed_accuracy=row["model_change_acc"]))
summary = pd.DataFrame(records)
# Scorer cross-check against a direct calculation on one matched slice.
sample = scored.loc[scored.lead_h.eq(1)]
reported = summary.query("method == 'extra_trees' and slice == 'all_recorded' and lead_h == 1").iloc[0]
assert np.isclose(reported.accuracy, ((sample.extra_trees >= .5) == sample.y_od).mean())
assert np.isclose(reported.brier, ((sample.extra_trees - sample.y_od) ** 2).mean())
summary.to_csv(OUT / "metrics.csv", index=False)
scored.to_parquet(OUT / "per_row_traces.parquet", index=False)
joblib.dump({"original": original, "alternate": alternate}, OUT / "models.joblib", compress=3)
manifest = {"source_commit": REV, "source_archive_sha256": hashlib.sha256(archive).hexdigest(), "experiment": "October fold; fixed settings chosen before alternative scores; not full six-month rerun", "feature_order": FEATURES, "train_rows": len(fit_rows), "test_rows": len(test_rows), "scored_rows": len(scored), "train_label_end": fit_rows.label_available_at.max().isoformat(), "test_issue_start": test_rows.issue_time.min().isoformat(), "test_issue_end": test_rows.issue_time.max().isoformat(), "skipped_issues": skipped, "mix": "0.5 original_blend + 0.5 extra_trees", "python": platform.python_version(), "sklearn": sklearn.__version__, "pandas": pd.__version__, "numpy": np.__version__, "elapsed_s": time.time()-started, "limitations": ["One month only; October already used in development, not a pristine final lockbox", "Same public proxy labels and assumed weather publication lag", "Retrospective stuck-feed exclusion is sensitivity only", "Overlapping forecast rows are correlated; no significance claim", "Runway 18 alternative not evaluated"], "file_sha256": {str(p.relative_to(ROOT)): hashlib.sha256(p.read_bytes()).hexdigest() for folder in (ROOT / "ml", ROOT / "data") for p in folder.rglob("*") if p.is_file() and "__pycache__" not in p.parts}}
(OUT / "manifest.json").write_text(json.dumps(manifest, indent=2, default=str))
print(summary.query("slice == 'all_recorded'").round(5).to_string(index=False))
print("EXPERIMENT_COMPLETE", round(time.time()-started, 1), "seconds", flush=True)

        method        slice  lead_h   n  accuracy    brier  changed_n  changed_accuracy
  original_raw all_recorded       1 105  0.819048 0.141913          4          0.250000
  original_raw all_recorded       6 107  0.803738 0.183150         12          0.750000
  original_raw all_recorded      24 107  0.757009 0.215326         33          0.727273
  original_raw all_recorded      48 107  0.728972 0.232775         52          0.711538
original_blend all_recorded       1 105  0.942857 0.060113          4          0.000000
original_blend all_recorded       6 107  0.813084 0.144555         12          0.750000
original_blend all_recorded      24 107  0.757009 0.215326         33          0.727273
original_blend all_recorded      48 107  0.728972 0.226940         52          0.711538
   extra_trees all_recorded       1 105  0.809524 0.082091          4          0.000000
   extra_trees all_recorded       6 107  0.794393 0.163646         12          0.750000
   extra_trees all_recorded     